# درس ۱۸: ایمن‌سازی عامل‌های هوش مصنوعی با رسیدهای رمزنگاری‌شده

## دفترچه آموزشی تعاملی

این دفترچه چهار تمرین را پوشش می‌دهد:

۱. **اولین رسید خود را امضا کنید** برای فراخوانی ابزار عامل و آن را تأیید کنید.
۲. **به رسید دستکاری کنید** و مشاهده کنید که تأیید شکست می‌خورد.
۳. **یک زنجیره سه‌رسیدی بسازید** و صحت زنجیره را تأیید کنید.
۴. **یک فراخوانی ابزار چارچوب عامل مایکروسافت را بسته‌بندی کنید** تا هر عمل یک رسید ایجاد کند.

تمام عناصر رمزنگاری از کتابخانه‌های به‌خوبی نگهداری‌شده وارد می‌شوند (`pynacl` برای Ed25519، `jcs` برای JSON رسمی RFC 8785، `hashlib` از کتابخانه استاندارد پایتون برای SHA-256). خود منطق رسید به زبان ساده پایتون است که می‌توانید بخوانید و اصلاح کنید.

سلول‌ها را به ترتیب اجرا کنید. هر بخش کوتاه و مستقل است.


## راه‌اندازی

دو وابستگی را نصب کنید. هر دو دارای مجوزهای باز (Apache-2.0 / MIT) هستند.


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ابزارهای کمکی

این دو ابزار کمکی عمل کدگذاری base64url (بدون پدینگ) و هش‌گذاری SHA-256 اشیاء دلخواه را انجام می‌دهند. آن‌ها باقی نوت‌بوک را متمرکز بر منطق رسید نگه می‌دارند.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## بخش ۱: رسید اول خود را امضا کنید

تصور کنید نماینده ما برای **Contoso Travel** به تازگی پروازهایی از سیدنی به لس‌آنجلس را برای یک مشتری جستجو کرده است. ما می‌خواهیم این تماس ابزار را به عنوان یک رسید امضا شده ثبت کنیم تا بازرس آینده بتواند بدون اعتماد به ما آن را تأیید کند.

### گام ۱.۱: تولید یک کلید امضا

در محیط تولید، کلید امضای نماینده در یک ماژول امنیتی سخت‌افزاری (HSM)، Azure Key Vault، یا یک مخزن محافظت‌شده مشابه نگهداری می‌شود. برای این درس، ما یک کلید تازه در حافظه تولید می‌کنیم.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### مرحله ۱.۲: ساخت محموله رسید

محموله شامل همه چیزهایی است که می‌خواهیم رسید به آن گواهی دهد: چه کسی عمل کرده، چه ابزاری، با چه آرگومان‌هایی، چه چیزی بازگشته، تحت چه سیاستی، و چه زمانی. ما به جای درج مستقیم آرگومان‌ها و نتیجه، آنها را هش می‌کنیم تا رسید اطلاعات حساس را فاش نکند.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### مرحله ۱.۳: امضا و مونتاژ رسید

سه مرحله:

۱. استانداردسازی محتوای پیام با استفاده از JCS به طوری که دو پیاده‌سازی که همان رسید منطقی را تولید می‌کنند، بایت‌های دقیقا یکسان تولید کنند.
۲. هش کردن بایت‌های استاندارد شده با SHA-256.
۳. امضای هش با کلید خصوصی Ed25519.

سپس امضا به پیام اصلی متصل شده و رسید نهایی تولید می‌شود.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### مرحله ۱.۴: تأیید رسید

تأیید روند را معکوس می‌کند. ما امضا را جدا می‌کنیم، هش اولیه را دوباره محاسبه می‌کنیم و امضا را با کلید عمومی موجود در رسید بررسی می‌کنیم.

یک حسابرس که این تأیید را انجام می‌دهد، هیچ چیزی از ما نمی‌خواهد جز خود رسید. نیازی به تماس با سرویس، پرس‌وجوی فهرست کلید یا اعتماد ندارد.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

باید ببینید `Receipt is valid: True`. عامل اولین رکورد ممیزی امضاشده رمزنگاری شده خود را تولید کرده است.


## بخش ۲: دستکاری رسید

همهٔ هدف رسیدها این است که دستکاری‌شدن در آن‌ها قابل تشخیص باشد. بیایید این را اثبات کنیم.

ما دقیقاً یک کاراکتر از رسید را تغییر خواهیم داد و خواهیم دید که تأییدیه شکست می‌خورد.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### چه چیزی اتفاق افتاد؟

وقتی `policy_id` را تغییر دادیم، بایت‌های اصلی تغییر کردند. هش SHA-256 آن بایت‌ها تغییر کرد. امضا (که بر روی هش اصلی بود) دیگر با هش جدید تطابق ندارد. تأیید صحت به درستی مقدار `False` را برمی‌گرداند.

هیچ راهی برای تغییر هیچ فیلدی از رسید و در عین حال تایید صحت آن وجود ندارد، مگر اینکه مهاجم کلید خصوصی را داشته باشد. تا زمانی که کلید خصوصی در یک مخزن کلید باشد و کلید عمومی منتشر شده باشد، دستکاری غیرممکن است که مخفی بماند.

خودتان امتحان کنید: `tool_name` یا `agent_id` یا `timestamp` را در سلول بالا تغییر دهید و دوباره اجرا کنید. هر تغییر منجر به رسید نامعتبر می‌شود.


## بخش ۳: رسیدها را به هم زنجیره کنید

یک رسید تنها از یک عمل محافظت می‌کند. اکثر عوامل چندین عمل انجام می‌دهند. برای اینکه کل دنباله قابل مشاهده برای تغییرات باشد، هر رسید را با درج هش رسید قبلی در بارگذاری رسید جدید به رسید قبلی متصل می‌کنیم.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

اگر کسی رسیدی را حذف یا جابجا کند، زنجیره دقیقاً در همان نقطه شکسته می‌شود. اعتبارسنجی هر رسید بعدی ناموفق خواهد بود زیرا `previous_receipt_hash` آن دیگر با هش واقعی پیش‌نیازش مطابقت ندارد.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

حالا زنجیره را با دستکاری رسید وسط بشکنید و دوباره بررسی کنید. رسید دستکاری‌شده در بررسی امضایش رد می‌شود، و رسید بعدی در بررسی لینک زنجیره‌ای خود رد می‌شود (چون `previous_receipt_hash` آن دیگر با هش رسید وسطی تغییر یافته مطابقت ندارد).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

رسید ۰ هنوز تأیید می‌شود (تغییری در آن ایجاد نشده و پیش‌نیازی برای وابستگی ندارد). رسید ۱ در چک امضای خود شکست می‌خورد زیرا ما `tool_args_hash` را تغییر دادیم. رسید ۲ در چک زنجیره‌ای خود شکست می‌خورد چون `previous_receipt_hash` آن بر اساس رسید اصلی (که اکنون تغییر کرده است) محاسبه شده بود.

حتی اگر یک مهاجم رسید ۱ تغییر یافته را مجدداً امضا کند (که بدون کلید خصوصی قادر به انجام آن نیست)، ناسازگاری زنجیره‌ای در رسید ۲ هنوز جعل را آشکار می‌کرد. برای پنهان کردن تغییر، مهاجم باید از نقطه تغییر به بعد هر رسید را مجدداً امضا کند که این نیازمند داشتن کلید خصوصی است.


## بخش ۴: پوشش فراخوانی ابزار عامل با امضای رسید

در یک استقرار واقعی، شما نمی‌خواهید هر نویسنده عامل به یاد داشته باشد که `make_receipt` را فراخوانی کند. شما می‌خواهید امضای رسید به طور خودکار برای هر فراخوانی ابزار انجام شود.

اینجا ساده‌ترین الگو آمده است: یک کلاس پوشش‌دهنده که هر تابع ابزار قابل فراخوانی را می‌گیرد و نسخه‌ای که رسید تولید می‌کند از آن بازمی‌گرداند. این الگو با هر چارچوب عاملی، از جمله چارچوب عامل مایکروسافت (`agent_framework.foundry`) سازگار است.

اگر پروژه Microsoft Foundry تنظیم نکرده‌اید، ماک محلی زیر هنوز این الگو را نشان می‌دهد.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### یکپارچه‌سازی با چارچوب Microsoft Agent

پوشش `ReceiptedTool` بالا مستقل از چارچوب است. برای استفاده از آن در داخل یک عامل ساخته شده با چارچوب Microsoft Agent، تابع پوشش داده شده را به عنوان یک ابزار ثبت کنید. یک طرح کلی (شما جایگزین نمونه به ثبت ابزار واقعی Microsoft Foundry خواهید کرد):

```python
# شبه‌کد نشان‌دهنده شکل یکپارچه‌سازی.
# وارد کردن os
# از agent_framework.foundry وارد کردن FoundryChatClient
# از azure.identity وارد کردن AzureCliCredential
#
# ارائه‌دهنده = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# عامل = ارائه‌دهنده.as_agent(
#     دستورالعمل‌ها="شما یک نماینده سفر Contoso هستید ...",
#     tools=[receipted_lookup],   # ابزار بسته‌بندی شده، نه تابع خام
# )
# پاسخ = عامل.run("پیدا کردن پروازها از سیدنی به لس‌آنجلس در ژوئن.")
#
# # پس از اجرا، هر فراخوانی ابزاری که عامل انجام داده، رسید امضا شده دارد:
# audit_chain = receipted_lookup.receipts
```

چارچوب عامل نیازی به دانستن هیچ چیزی درباره رسیدها ندارد. امضای رسید به دور ابزار پیچیده شده است، نه اینکه به چارچوب متصل شود. این روش افزودن منشاء به کد عامل موجود بدون بازنویسی عامل است.


## بازبینی و چالش کششی

شما:

- یک جفت کلید Ed25519 تولید کرده‌اید.
- رسیدی برای یک فراخوانی ابزار عامل ساخته و امضا کرده‌اید.
- رسید را به‌صورت آفلاین تنها با استفاده از کلید عمومی تأیید اعتبار کرده‌اید.
- با رسید دستکاری کرده و شاهد شکست تأیید اعتبار بوده‌اید.
- یک زنجیره هش‌شده شامل سه رسید ساخته‌اید.
- وسط زنجیره را دستکاری کرده و هر دو شکست امضا و شکست پیوند زنجیره را مشاهده کرده‌اید.
- یک تابع ابزار را با امضای خودکار رسید پوشانده‌اید.

**چالش کششی.** اسکیمای رسید را با یک فیلد `request_id` (یک UUID برای ردیابی توزیعی) گسترش دهید. `make_receipt` را برای گنجاندن آن به‌روزرسانی کنید و تایید کنید که رسیدها هنوز از ابتدا تا انتها صحت دارند. سپس پس از امضا فیلد را تغییر دهید و تایید اعتبار شکست بخورد. این شما را مجبور می‌کند که درک کنید چگونه هر بایت از کدگذاری کاننیکال به امضا کمک می‌کند.

**مرز مهم.** رسیدها سه چیز و فقط سه چیز را اثبات می‌کنند: نسبت‌دهی (این کلید این محتوا را امضا کرده است)، تمامیت (محتوا از زمان امضا تغییر نکرده است)، و ترتیب (این رسید پس از آن رسید آمده است). آنها اثبات نمی‌کنند که اقدام عامل درست بوده، سیاست نام‌برده‌شده در `policy_id` واقعاً ارزیابی شده باشد، یا اینکه عامل هر قانون را دنبال کرده باشد. رسیدها یک بنیان هستند. حاکمیت سیستمی است که روی آن می‌سازید.

README درس را دوباره با در نظر گرفتن این مرز بخوانید. رایج‌ترین اشتباه تیم‌ها در مورد رسیدها این است که فرض می‌کنند "ما رسید داریم" یعنی "ما حکمرانی داریم." این درست نیست. رسیدها رفتار عامل را قابل حسابرسی می‌کنند. آنها آن را درست نمی‌کنند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
